# 04 — FOMC Knowledge Graphs

Interactive exploration of causal knowledge graphs built from FOMC meeting minutes.

This notebook:
1. Builds standardized knowledge graphs per economic period
2. Visualizes causal relationships between economic concepts
3. Compares network structure across periods
4. Enables interactive exploration of specific concepts and relationships

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle

from src.config import ROOT
from src.standardize_terms import standardize_term

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

## 1. Build Knowledge Graphs

Run the graph building pipeline to create standardized graphs from faithful causal triples.

In [ ]:
# Build graphs (or load if already built)
from src import build_knowledge_graph

# Run the builder
graphs = build_knowledge_graph.main()

## 2. Generate Visualizations

Create network visualizations for each period.

In [ ]:
from src import visualize_knowledge_graph

# Generate visualizations
visualize_knowledge_graph.main()

## 3. Display Visualizations

In [ ]:
from IPython.display import Image, display

periods = [
    ('great_moderation', 'Great Moderation (1994-2007)'),
    ('post_crisis_zlb', 'Post-Crisis ZLB (2008-2019)'),
    ('post_covid', 'Post-COVID Surge (2020-2023)')
]

for period_key, period_name in periods:
    print(f"\n{'='*70}")
    print(f"{period_name}")
    print(f"{'='*70}")
    img_path = ROOT / 'outputs' / f'kg_{period_key}.png'
    display(Image(filename=str(img_path), width=1000))

## 4. Graph Statistics

Compute and display comprehensive statistics for each period.

In [ ]:
# Load statistics
stats_df = pd.read_csv(ROOT / 'outputs' / 'graph_statistics.csv')

print("Cross-Period Comparison:")
print("="*70)
display(stats_df)

In [ ]:
# Visualize key metrics across periods
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Knowledge Graph Metrics Across Economic Periods', fontsize=14, fontweight='bold')

metrics = [
    ('nodes', 'Number of Nodes'),
    ('edges', 'Number of Edges'),
    ('density', 'Graph Density'),
    ('avg_degree', 'Average Degree'),
    ('avg_edge_weight', 'Average Edge Weight'),
    ('largest_component_pct', 'Largest Component (%)'),
]

for idx, (metric, title) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    ax.bar(stats_df['period'], stats_df[metric], color=['#3498db', '#e74c3c', '#2ecc71'])
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Top Concepts by Period

Examine the most central concepts in each period.

In [ ]:
for period_key, period_name in periods:
    print(f"\n{'='*70}")
    print(f"{period_name} - Top 10 Concepts")
    print(f"{'='*70}")
    
    top_nodes = pd.read_csv(ROOT / 'outputs' / f'kg_top_nodes_{period_key}.csv', index_col=0)
    display(top_nodes)

## 6. Top Causal Relationships by Period

Examine the strongest causal relationships in each period.

In [ ]:
for period_key, period_name in periods:
    print(f"\n{'='*70}")
    print(f"{period_name} - Top 10 Causal Relationships")
    print(f"{'='*70}")
    
    top_edges = pd.read_csv(ROOT / 'outputs' / f'kg_top_edges_{period_key}.csv', index_col=0)
    display(top_edges)

## 7. Interactive Exploration

Load graphs for interactive querying.

In [ ]:
# Load all graphs
graphs = {}
for period_key, _ in periods:
    graph_path = ROOT / 'outputs' / 'knowledge_graphs' / f'graph_{period_key}.pkl'
    with open(graph_path, 'rb') as f:
        graphs[period_key] = pickle.load(f)

print("Loaded graphs:")
for period, G in graphs.items():
    print(f"  {period}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

### 7.1 Find What Causes a Concept

In [ ]:
def find_causes(concept: str, period: str = 'post_covid'):
    """Find all concepts that cause the given concept."""
    G = graphs[period]
    
    if concept not in G.nodes():
        print(f"Concept '{concept}' not found in {period} graph.")
        return
    
    causes = []
    for cause, effect, data in G.in_edges(concept, data=True):
        causes.append({
            'cause': cause,
            'weight': data['weight'],
            'frequency': data['frequency'],
            'examples': data['examples'][0] if data['examples'] else ''
        })
    
    if not causes:
        print(f"No causes found for '{concept}' in {period}.")
        return
    
    df = pd.DataFrame(causes).sort_values('weight', ascending=False)
    print(f"\nWhat causes '{concept}' in {period}:")
    display(df)

# Example: What causes inflation in post-COVID period?
find_causes('inflation', 'post_covid')

### 7.2 Find What a Concept Causes

In [ ]:
def find_effects(concept: str, period: str = 'post_covid'):
    """Find all concepts caused by the given concept."""
    G = graphs[period]
    
    if concept not in G.nodes():
        print(f"Concept '{concept}' not found in {period} graph.")
        return
    
    effects = []
    for cause, effect, data in G.out_edges(concept, data=True):
        effects.append({
            'effect': effect,
            'weight': data['weight'],
            'frequency': data['frequency'],
            'examples': data['examples'][0] if data['examples'] else ''
        })
    
    if not effects:
        print(f"No effects found for '{concept}' in {period}.")
        return
    
    df = pd.DataFrame(effects).sort_values('weight', ascending=False)
    print(f"\nWhat '{concept}' causes in {period}:")
    display(df)

# Example: What does monetary policy cause in post-COVID period?
find_effects('monetary_policy', 'post_covid')

### 7.3 Compare a Concept Across Periods

In [ ]:
def compare_concept_across_periods(concept: str):
    """Compare a concept's presence and centrality across all periods."""
    comparison = []
    
    for period_key, period_name in periods:
        G = graphs[period_key]
        
        if concept in G.nodes():
            degree = G.degree(concept)
            in_deg = G.in_degree(concept)
            out_deg = G.out_degree(concept)
            centrality = nx.degree_centrality(G)[concept]
            
            comparison.append({
                'period': period_name,
                'present': 'Yes',
                'degree': degree,
                'in_degree': in_deg,
                'out_degree': out_deg,
                'centrality': centrality,
            })
        else:
            comparison.append({
                'period': period_name,
                'present': 'No',
                'degree': 0,
                'in_degree': 0,
                'out_degree': 0,
                'centrality': 0,
            })
    
    df = pd.DataFrame(comparison)
    print(f"\nConcept '{concept}' across periods:")
    display(df)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].bar(df['period'], df['degree'], color=['#3498db', '#e74c3c', '#2ecc71'])
    axes[0].set_title(f"Degree of '{concept}'", fontweight='bold')
    axes[0].set_ylabel('Degree')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].bar(df['period'], df['centrality'], color=['#3498db', '#e74c3c', '#2ecc71'])
    axes[1].set_title(f"Centrality of '{concept}'", fontweight='bold')
    axes[1].set_ylabel('Degree Centrality')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Example: Compare inflation across periods
compare_concept_across_periods('inflation')

### 7.4 Find Shortest Path Between Two Concepts

In [ ]:
def find_causal_path(cause: str, effect: str, period: str = 'post_covid'):
    """Find shortest causal path from cause to effect."""
    G = graphs[period]
    
    if cause not in G.nodes():
        print(f"Cause '{cause}' not found in {period} graph.")
        return
    if effect not in G.nodes():
        print(f"Effect '{effect}' not found in {period} graph.")
        return
    
    try:
        path = nx.shortest_path(G, cause, effect)
        print(f"\nCausal path from '{cause}' to '{effect}' in {period}:")
        print(" → ".join(path))
        
        # Show edge details
        print("\nEdge details:")
        for i in range(len(path) - 1):
            u, v = path[i], path[i+1]
            data = G[u][v]
            print(f"  {u:30} → {v:30} (weight={data['weight']:.1f}, freq={data['frequency']})")
            
    except nx.NetworkXNoPath:
        print(f"No path found from '{cause}' to '{effect}' in {period}.")

# Example: Path from monetary policy to economic activity
find_causal_path('monetary_policy', 'economic_activity', 'post_covid')

## 8. Cross-Period Concept Analysis

Which concepts are unique to each period vs. shared?

In [ ]:
# Get nodes for each period
nodes_by_period = {
    period_key: set(G.nodes())
    for period_key, G in graphs.items()
}

# Find shared and unique concepts
all_concepts = set.union(*nodes_by_period.values())
shared_all = set.intersection(*nodes_by_period.values())

print(f"Total unique concepts across all periods: {len(all_concepts)}")
print(f"Concepts appearing in all periods: {len(shared_all)}")

print("\nShared concepts (top 20 by total degree):")
shared_degrees = {}
for concept in shared_all:
    total_degree = sum(graphs[p].degree(concept) for p in graphs.keys())
    shared_degrees[concept] = total_degree

for i, (concept, deg) in enumerate(sorted(shared_degrees.items(), key=lambda x: x[1], reverse=True)[:20], 1):
    print(f"  {i:2}. {concept:30} (total degree={deg})")

# Period-unique concepts
print("\nPeriod-unique concepts:")
for period_key, period_name in periods:
    other_periods = [p for p in graphs.keys() if p != period_key]
    other_nodes = set.union(*[nodes_by_period[p] for p in other_periods])
    unique = nodes_by_period[period_key] - other_nodes
    print(f"  {period_name}: {len(unique)} unique concepts")
    if unique:
        # Show top 5 by degree
        unique_sorted = sorted(unique, key=lambda c: graphs[period_key].degree(c), reverse=True)[:5]
        for concept in unique_sorted:
            deg = graphs[period_key].degree(concept)
            print(f"    - {concept} (degree={deg})")

## 9. Summary

Key findings from the knowledge graph analysis:

- **Great Moderation**: Focus on traditional monetary policy → growth relationships
- **Post-Crisis ZLB**: Energy prices become major inflation driver, unconventional policy language
- **Post-COVID**: Inflation becomes highly central with self-reinforcing loops, supply chain emphasis